# Guía de comandos y conceptos — Panadería, Grupo 4 (Power Rangers)

Este cuaderno recorre lo que se hizo en el proyecto y los comandos que sirven como **evidencia** en la lista de cotejo (secciones A a G). Cada sección explica el concepto, da el comando y dice qué se debe ver.

**Cómo está escrito**

- **Cada celda es una sola línea.** Se copia, se pega en Cloud Shell y se ejecuta. No hay variables compartidas entre celdas: los valores (proyecto, namespace, imágenes) están escritos tal cual.
- Se ejecuta desde la carpeta del repositorio: `cd ~/Virtualizacion_Proyecto_Final_PW`.
- Las direcciones IP se piden dentro de la misma línea con `$(kubectl get svc ...)`, así que no hay que copiarlas a mano.
- Las celdas con ⚠️ **cambian o borran cosas**. Léelas antes de ejecutarlas.
- Está pensado para **estudiar y ensayar**. En la defensa no se permite leer guiones ni ejecutar scripts preparados: los comandos se escriben en vivo, así que conviene practicarlos hasta saberlos de memoria.
- Cada captura del PDF de evidencias debe mostrar el **reloj del sistema** y la salida de `gcloud config list` con el proyecto.

| Sección | Qué se demuestra |
|---|---|
| A | Repositorio, README, diagrama, carnés, nombres del equipo |
| B | Microservicios, Dockerfiles y frontend |
| C | Imágenes en Artifact Registry |
| D | Terraform: plan, apply, nodos spot |
| E | Despliegue: Deployments, ConfigMap, Secret, PVC, IP pública, DNS |
| F | Operación: escalar, recrear pods, rolling update, logs |
| G | Costos y destrucción |

## 1. Identidad del proyecto y reloj (en **todas** las capturas)

`gcloud config list` muestra la cuenta y el proyecto activos. **Debe verse:** la fecha de hoy y `project = virtualizacion-proyecto-pr`.

In [ ]:
date; gcloud config list

Si el proyecto no coincide, se cambia con:

In [ ]:
gcloud config set project virtualizacion-proyecto-pr

## 2. Pre-vuelo (sección 8 del enunciado)

Antes de escribir código se confirma que la cuenta, la facturación, las APIs y Terraform están listos.

In [ ]:
gcloud auth list

In [ ]:
gcloud billing projects describe virtualizacion-proyecto-pr --format="value(billingEnabled)"

In [ ]:
gcloud services list --enabled --filter="config.name:(container.googleapis.com OR artifactregistry.googleapis.com OR compute.googleapis.com OR cloudresourcemanager.googleapis.com)" --format="value(config.name)"

In [ ]:
terraform version | head -1

**Debe verse:** `True` en facturación, las cuatro APIs y la versión de Terraform.

**Concepto:** Terraform necesita las APIs habilitadas antes de crear nada; leer datos del proyecto con la API de Resource Manager apagada produce `accessNotConfigured`. Si falta alguna:

In [ ]:
gcloud services enable container.googleapis.com artifactregistry.googleapis.com compute.googleapis.com cloudresourcemanager.googleapis.com

## 3. Sección A — Repositorio y trazabilidad

**Concepto:** el enunciado pide al menos 10 confirmaciones (commits) repartidas en 3 días distintos, sin alterar fechas. **Debe verse:** más de 10 commits y al menos 3 fechas.

In [ ]:
git rev-list --count HEAD

In [ ]:
git log --format=%ad --date=short | sort | uniq -c

In [ ]:
git log --oneline -5

### Identificación del equipo (A5) y carnés (A4)

El clúster y el namespace llevan el identificador del equipo, y los carnés están en la base de datos y en la interfaz. **Debe verse:** el namespace `panaderia-grupo-4` y 5 filas con los carnés (si no hay filas, faltan los datos iniciales: sección 8).

In [ ]:
kubectl get ns panaderia-grupo-4

In [ ]:
kubectl exec -n panaderia-grupo-4 deploy/postgres -- sh -c "PGPASSWORD=\$POSTGRES_PASSWORD psql -U panaderia_admin -d productos_db -c \"select nombre from productos where nombre like '%Carné%' order by nombre\""

## 4. Sección B — Los microservicios

**Concepto:** cada entidad tiene su propio servicio, su propia base lógica y su propio Dockerfile. Pedidos referencia a Productos solo por el identificador, sin clave foránea, porque las bases están separadas.

**Debe verse:** tres Dockerfiles; en el de Productos, dos etapas `FROM ... AS` y la línea `USER` (el contenedor no corre como administrador).

In [ ]:
ls services/*/Dockerfile

In [ ]:
grep -n "^FROM\|^USER" services/productos/Dockerfile

### CRUD de ambos servicios (B1 y B2)

Cada comando usa las IP públicas del clúster.

In [ ]:
curl -s "http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos?limit=2"

In [ ]:
curl -s -X POST http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos -H "content-type: application/json" -d '{"nombre":"Pan de demostración","precio":"4.50"}'

Para consultar, actualizar y borrar un producto se necesita su `id` (cópialo de la respuesta anterior y reemplaza `<ID>`):

In [ ]:
curl -s http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos/<ID>

In [ ]:
curl -s -X PATCH http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos/<ID> -H "content-type: application/json" -d '{"precio":"5.00"}'

In [ ]:
curl -s -o /dev/null -w "HTTP %{http_code}\n" -X DELETE http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos/<ID>

Crear un pedido con el primer producto del catálogo y listar pedidos (el total lo calcula el servidor):

In [ ]:
curl -s -X POST http://$(kubectl get svc pedidos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/pedidos -H "content-type: application/json" -d "{\"producto_id\":\"$(curl -s "http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos?limit=1" | python3 -c "import sys,json;print(json.load(sys.stdin)['items'][0]['id'])")\",\"cantidad\":3}" 

In [ ]:
curl -s "http://$(kubectl get svc pedidos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/pedidos?limit=2"

**Concepto:** el cliente solo envía `producto_id` y `cantidad`. El precio y el total los calcula el servidor; si el cliente envía `total`, la API responde `422`.

### Regla de integridad: no borrar un producto con pedidos

Productos pregunta a Pedidos por HTTP antes de borrar. La línea siguiente crea un producto, le pone un pedido e intenta borrarlo. **Debe verse:** `409` con el código `product_has_orders`.

*Esta línea usa una variable `ID` solo dentro de sí misma; no queda nada guardado.*

In [ ]:
ID=$(curl -s -X POST http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos -H "content-type: application/json" -d '{"nombre":"Pan con pedido","precio":"3.00"}' | python3 -c "import sys,json;print(json.load(sys.stdin)['id'])"); curl -s -X POST http://$(kubectl get svc pedidos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/pedidos -H "content-type: application/json" -d "{\"producto_id\":\"$ID\",\"cantidad\":2}" > /dev/null; curl -s -w "\nHTTP %{http_code}\n" -X DELETE http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos/$ID

**Limpieza** (⚠️ borra los datos de la prueba): primero el pedido, después el producto.

In [ ]:
for i in $(curl -s "http://$(kubectl get svc pedidos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/pedidos?limit=100" | python3 -c "import sys,json;[print(p['id']) for p in json.load(sys.stdin)['items'] if p['producto_nombre']=='Pan con pedido']"); do curl -s -o /dev/null -w "pedido HTTP %{http_code}\n" -X DELETE http://$(kubectl get svc pedidos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/pedidos/$i; done

In [ ]:
for i in $(curl -s "http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos?limit=100" | python3 -c "import sys,json;[print(p['id']) for p in json.load(sys.stdin)['items'] if p['nombre']=='Pan con pedido']"); do curl -s -o /dev/null -w "producto HTTP %{http_code}\n" -X DELETE http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos/$i; done

**Debe verse:** `pedido HTTP 204` y luego `producto HTTP 204`. *Pregunta típica:* ¿por qué no basta una clave foránea? Porque cada servicio tiene su propia base de datos.

## 5. Sección C — Imágenes en Artifact Registry

**Concepto:** una imagen se identifica por la ruta completa `región-docker.pkg.dev/proyecto/repositorio/imagen:etiqueta`. Los manifiestos solo pueden referenciar imágenes de ahí (C3).

### Construir y publicar (C1)

⚠️ Tarda varios minutos. Desde Cloud Shell, Docker Hub agota el tiempo de espera, por eso el script descarga las imágenes base del espejo de Google. Los envíos al registro pueden fallar de forma intermitente con `connection refused`; el script reintenta.

In [ ]:
bash scripts/publicar_imagenes.sh

### Listar el repositorio (C2)

**Debe verse:** `frontend`, `pedidos` y `productos` con `1.1.0`, `productos` también con `1.1.1`, y `postgres` con `16`.

In [ ]:
gcloud artifacts docker images list us-central1-docker.pkg.dev/virtualizacion-proyecto-pr/panaderia-repo --include-tags --format="table(package.basename(), tags, createTime)"

### Imágenes que usan los manifiestos (C3)

**Debe verse:** todas empiezan con `us-central1-docker.pkg.dev/virtualizacion-proyecto-pr/panaderia-repo/`.

In [ ]:
grep -h "image:" deploy/k8s/*.yaml | sort -u

## 6. Sección D — Infraestructura como código con Terraform

**Concepto:** Terraform describe la infraestructura en archivos `.tf` y la crea de forma repetible. No gestiona los objetos de Kubernetes, que se aplican con `kubectl`.

| Archivo | Qué declara |
|---|---|
| `provider.tf` | El proveedor de Google Cloud |
| `variables.tf` | `project_id`, `region`, `zone`, `cluster_name`, sin valores sensibles |
| `vpc.tf` | Red y subred con rangos para pods y Services |
| `gke.tf` | Clúster GKE zonal y node pool de 2 nodos `e2-small` spot |
| `artifact_registry.tf` | Repositorio Docker y permiso de lectura para los nodos |
| `outputs.tf` | Nombre del clúster, comando de conexión y ruta del registro |

### Variables y valores (D2)

**Debe verse:** cuatro variables sin contraseñas ni claves, y un `terraform.tfvars` con solo el proyecto, la región, la zona y el nombre del clúster.

In [ ]:
ls infra/terraform/*.tf infra/terraform/terraform.tfvars.example

In [ ]:
grep -E '^variable|default' infra/terraform/variables.tf

In [ ]:
cat infra/terraform/terraform.tfvars

### Inicializar y planificar (D3)

`init` descarga el proveedor. `plan` muestra qué se va a crear **sin crear nada**; su captura se hace **antes** del `apply`. **Debe verse:** `Plan: 6 to add, 0 to change, 0 to destroy.` (repositorio, permiso de lectura, red, subred, clúster y node pool).

In [ ]:
terraform -chdir=infra/terraform init

In [ ]:
terraform -chdir=infra/terraform plan -out=tfplan

### Aplicar (D4)

⚠️ **Crea recursos que cuestan dinero.** El clúster tarda unos 8 minutos. **Debe verse:** `Apply complete! Resources: 6 added, 0 changed, 0 destroyed.`

In [ ]:
date; terraform -chdir=infra/terraform apply tfplan

In [ ]:
terraform -chdir=infra/terraform output

### Nodos spot (D5)

Los nodos spot son máquinas con descuento que Google puede reclamar. **Debe verse:** `e2-small`, `spot: True` y 2 nodos `Ready`.

In [ ]:
gcloud container node-pools list --cluster panaderia-grupo-4 --zone us-central1-a --format="table(name,config.machineType,config.spot,initialNodeCount)"

In [ ]:
kubectl get nodes

## 7. Conectar `kubectl` al clúster

`get-credentials` escribe en `~/.kube/config` las credenciales del clúster.

In [ ]:
gcloud container clusters get-credentials panaderia-grupo-4 --zone us-central1-a --project virtualizacion-proyecto-pr

In [ ]:
kubectl config current-context

## 8. Sección E — Despliegue en GKE

**Concepto:** los objetos de Kubernetes están en `deploy/k8s/` y se aplican **en orden**: namespace, ConfigMap, Secret, volumen y base de datos, migraciones y, por último, los servicios.

### Crear el Secret

El script genera las contraseñas al azar y **no las imprime**. Si el Secret ya existe, no lo cambia.

In [ ]:
bash scripts/crear_secret_gke.sh

### Desplegar

⚠️ Aplica los manifiestos y espera a que cada pieza esté lista.

In [ ]:
bash deploy/k8s/apply.sh

### Estado de todos los recursos (E1)

**Debe verse:** pods de `frontend`, `pedidos`, `productos` y `postgres` en `Running`; los dos Jobs de migración en `Completed`; tres Services `LoadBalancer` con IP externa; el PVC `postgres-pvc` en `Bound`.

Si una IP dice `<pending>`, espera de 1 a 3 minutos; si persiste, revisa la cuota de direcciones IP de la región.

In [ ]:
date; kubectl get all,pvc -n panaderia-grupo-4

### Cargar los datos iniciales y los carnés (A4)

El script es idempotente: si los productos ya existen, no los duplica. *Las dos variables antepuestas son los parámetros que el script lee y solo viven durante ese comando.*

In [ ]:
PRODUCTOS_URL=http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000 PEDIDOS_URL=http://$(kubectl get svc pedidos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000 python3 scripts/seed_database.py

### ConfigMap: configuración no sensible (E2)

**Concepto:** un ConfigMap guarda valores que no son secretos (URLs, tiempos). Los pods los reciben como variables de entorno, así el mismo contenedor funciona en cualquier entorno. **Debe verse:** el ConfigMap con sus claves y las mismas variables dentro del pod.

In [ ]:
kubectl get configmap panaderia-config -n panaderia-grupo-4 -o yaml

In [ ]:
kubectl exec -n panaderia-grupo-4 deploy/pedidos -- printenv PRODUCTOS_BASE_URL PRODUCTOS_TIMEOUT_SECONDS

### Secret: credenciales (E3)

**Concepto:** un Secret guarda credenciales. Se almacenan en base64, que **no es cifrado**; sirve para no escribirlas en los manifiestos ni en Git. **Debe verse:** cinco claves con su tamaño y la cadena de conexión con la contraseña como `********`. En la captura **no** muestres la contraseña real.

In [ ]:
kubectl describe secret panaderia-secrets -n panaderia-grupo-4

In [ ]:
kubectl exec -n panaderia-grupo-4 deploy/productos -- printenv DATABASE_URL | sed -E 's#(://[^:]+:)[^@]+@#\1********@#'

### Volumen persistente: los datos sobreviven al pod (E4)

**Concepto:** un pod es desechable; su disco interno desaparece con él. El PersistentVolumeClaim reserva un disco aparte, montado en la ruta de datos de PostgreSQL, que sobrevive.

La prueba tiene cuatro pasos: contar productos, **borrar el pod de PostgreSQL**, esperar el pod nuevo y contar de nuevo. **Debe verse:** el mismo número antes y después, y un pod nuevo con otro nombre. Si los datos se pierden, el volumen no está montado en la ruta de datos del motor; revisa el `mountPath` en `04-postgres.yaml`.

In [ ]:
kubectl get pvc -n panaderia-grupo-4

In [ ]:
curl -s "http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos?limit=1" | python3 -c "import sys,json;print(json.load(sys.stdin)['total'])"

In [ ]:
kubectl delete pod -n panaderia-grupo-4 -l app.kubernetes.io/name=postgres

In [ ]:
kubectl wait -n panaderia-grupo-4 -l app.kubernetes.io/name=postgres --for=condition=ready pod --timeout=240s

In [ ]:
curl -s "http://$(kubectl get svc productos -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}'):8000/productos?limit=1" | python3 -c "import sys,json;print(json.load(sys.stdin)['total'])"

In [ ]:
kubectl get pods -n panaderia-grupo-4 -l app.kubernetes.io/name=postgres

### IP pública (E5)

**Concepto:** un Service de tipo `LoadBalancer` pide a Google una dirección IP pública que reparte el tráfico entre los pods. **Debe verse:** el frontend con IP externa y código `200`.

**Importante:** la dirección es `http://`, no `https://`. Algunos navegadores, como Brave, la convierten a HTTPS y aparece `ERR_CONNECTION_TIMED_OUT`. La captura E5.2 es la página en el navegador, **fuera del clúster**, con la barra de direcciones visible.

In [ ]:
kubectl get svc -n panaderia-grupo-4

In [ ]:
curl -s -o /dev/null -w "HTTP %{http_code}\n" http://$(kubectl get svc frontend -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}')/

In [ ]:
echo "Abre en el navegador: http://$(kubectl get svc frontend -n panaderia-grupo-4 -o jsonpath='{.status.loadBalancer.ingress[0].ip}')"

### Comunicación interna por nombre DNS (E6)

**Concepto:** los pods cambian de IP cada vez que se recrean; por eso se usa el nombre del Service (`productos`, `pedidos`), que Kubernetes resuelve por DNS interno. **Debe verse:** `{"status":"ok"}` dos veces, líneas `proxy_pass http://productos:8000/...` y la URL con `.svc.cluster.local`.

In [ ]:
kubectl exec -n panaderia-grupo-4 deploy/frontend -- wget -q -O - http://productos:8000/health/live

In [ ]:
kubectl exec -n panaderia-grupo-4 deploy/frontend -- wget -q -O - http://pedidos:8000/health/live

In [ ]:
kubectl exec -n panaderia-grupo-4 deploy/frontend -- grep proxy_pass /etc/nginx/nginx.conf

In [ ]:
kubectl exec -n panaderia-grupo-4 deploy/pedidos -- printenv PRODUCTOS_BASE_URL

## 9. Sección F — Operación de Kubernetes

### Escalar a 3 réplicas de forma declarativa (F1)

**Concepto:** "declarativo" significa **cambiar el archivo** y aplicarlo, en vez de dar una orden puntual; el estado deseado queda escrito en el YAML.

⚠️ La primera línea cambia `06-productos.yaml` en esta copia local y lo aplica. **Debe verse:** `replicas: 3`, `deployment.apps/productos configured` y tres pods `Running`.

In [ ]:
sed -i 's/replicas: 2/replicas: 3/' deploy/k8s/06-productos.yaml && grep -n replicas deploy/k8s/06-productos.yaml && kubectl apply -f deploy/k8s/06-productos.yaml

In [ ]:
kubectl rollout status deployment/productos -n panaderia-grupo-4

In [ ]:
kubectl get pods -n panaderia-grupo-4 -l app.kubernetes.io/name=productos

In [ ]:
kubectl get deployment productos -n panaderia-grupo-4

Para volver a 2 réplicas (restaura el archivo y lo aplica):

In [ ]:
git checkout deploy/k8s/06-productos.yaml && kubectl apply -f deploy/k8s/06-productos.yaml

### Eliminar un pod y verlo recrear (F2)

**Concepto:** el Deployment vigila que haya el número de réplicas declarado. Si un pod desaparece, crea otro; a esto se le llama **autorrecuperación**.

⚠️ La primera línea borra un pod de Productos (los otros siguen atendiendo). **Debe verse:** un pod con edad de segundos y nombre distinto, hasta volver a `1/1 Running`.

In [ ]:
kubectl delete pod -n panaderia-grupo-4 $(kubectl get pods -n panaderia-grupo-4 -l app.kubernetes.io/name=productos -o name | head -1)

In [ ]:
kubectl get pods -n panaderia-grupo-4 -l app.kubernetes.io/name=productos -w

`-w` sigue los cambios en vivo; se detiene con `Ctrl+C`.

### Actualización de versión con rolling update (F3)

**Concepto:** un *rolling update* cambia la versión poco a poco: crea un pod nuevo, espera a que esté listo y recién entonces retira uno viejo. La estrategia es `maxSurge: 1` y `maxUnavailable: 0`.

⚠️ Cambia la imagen de Productos de `1.1.0` a `1.1.1`; la etiqueta `1.1.1` debe estar publicada (sección 5). **Debe verse:** pods viejos y nuevos conviviendo, `successfully rolled out` y la imagen `productos:1.1.1`.

In [ ]:
kubectl get deployment productos -n panaderia-grupo-4 -o jsonpath='{.spec.strategy}'

In [ ]:
kubectl set image deployment/productos productos=us-central1-docker.pkg.dev/virtualizacion-proyecto-pr/panaderia-repo/productos:1.1.1 -n panaderia-grupo-4

In [ ]:
kubectl rollout status deployment/productos -n panaderia-grupo-4

In [ ]:
kubectl get deployment productos -n panaderia-grupo-4 -o jsonpath='{.spec.template.spec.containers[0].image}'

In [ ]:
kubectl rollout history deployment/productos -n panaderia-grupo-4

Para volver atrás:

In [ ]:
kubectl rollout undo deployment/productos -n panaderia-grupo-4

**Nota:** en una prueba, durante el reinicio algunas solicitudes externas no recibieron respuesta, aunque la estrategia no permite bajar del mínimo. Conviene conocer el tema por si lo preguntan (`docs/cambios-pendientes.md`, P1).

### Registros de un pod (F4)

**Concepto:** cada contenedor escribe su salida estándar y Kubernetes la guarda; `kubectl logs` la lee sin entrar al pod. **Debe verse:** líneas de acceso de FastAPI (`GET /health/ready ... 200 OK`). Útiles: `-f` para seguir en vivo y `--previous` para ver el contenedor anterior si se reinició.

In [ ]:
kubectl logs deployment/productos -n panaderia-grupo-4 --tail=15

In [ ]:
kubectl logs deployment/pedidos -n panaderia-grupo-4 --tail=15

## 10. Diagnóstico de fallas (pregunta individual)

La defensa incluye una pregunta de diagnóstico. El método es siempre el mismo: **ver el estado, describir el objeto, leer los logs**.

| Síntoma | Causa probable | Cómo comprobarlo |
|---|---|---|
| `ImagePullBackOff` / `ErrImagePull` | Etiqueta inexistente, ruta equivocada o nodos sin permiso de lectura al registro (error `403`) | `kubectl describe pod` y leer el evento `Failed`; revisar el rol `artifactregistry.reader` de la cuenta de servicio de los nodos |
| `CrashLoopBackOff` | La aplicación falla al arrancar: variable faltante o base inaccesible | `kubectl logs <pod> --previous` |
| PVC en `Pending` | No hay StorageClass por defecto | `kubectl get storageclass` |
| Se pierden los datos al recrear PostgreSQL | El volumen no está montado en la ruta de datos | Revisar `mountPath` y `subPath` del Deployment |
| `EXTERNAL-IP` en `<pending>` | El balanceador tarda 1 a 3 minutos, o falta cuota de IP | `kubectl describe svc <nombre>` |
| El frontend no alcanza los servicios | Se usó una IP de pod en vez del nombre del Service | Usar `http://productos:8000` |
| Migración que no termina | Contraseña distinta en el Secret o la base no existe | `kubectl logs job/productos-migration-job` |
| `terraform apply` falla por una API | La API no está habilitada | `gcloud services enable ...` y repetir |
| `terraform destroy` deja recursos vivos | Se crearon a mano, fuera del estado | Eliminarlos aparte |
| La página no carga en el navegador | El navegador convirtió la dirección a `https://` | Escribir `http://` |
| `409` al borrar un producto | Tiene pedidos asociados | Borrar los pedidos o usar `activo=false` |

Comandos para investigar rápido:

In [ ]:
kubectl get pods -n panaderia-grupo-4

In [ ]:
kubectl describe pod <NOMBRE_DEL_POD> -n panaderia-grupo-4

In [ ]:
kubectl logs <NOMBRE_DEL_POD> -n panaderia-grupo-4 --previous

In [ ]:
kubectl get events -n panaderia-grupo-4 --sort-by=.lastTimestamp

In [ ]:
kubectl get pods -n panaderia-grupo-4 --field-selector=status.phase!=Running

## 11. Preguntas conceptuales — respuestas cortas

- **¿Qué es un Pod?** La unidad más pequeña de Kubernetes: uno o más contenedores que comparten red y almacenamiento. Es desechable.
- **¿Qué es un Deployment?** El objeto que declara cuántas réplicas de un pod debe haber y cómo actualizarlas. Si un pod muere, crea otro.
- **¿Qué es un Service?** Una dirección estable (nombre DNS e IP) delante de un grupo de pods. `ClusterIP` es interna; `LoadBalancer` agrega una IP pública.
- **¿ConfigMap o Secret?** El ConfigMap guarda configuración no sensible; el Secret guarda credenciales. Ninguno cifra por sí solo: base64 no es cifrado.
- **¿Para qué sirve el PVC?** Reserva un disco independiente del pod, para que los datos de PostgreSQL sobrevivan a su recreación.
- **¿Qué es un namespace?** Una división lógica del clúster. Aquí aísla todo en `panaderia-grupo-4`.
- **¿Qué hace Terraform y qué no?** Crea la infraestructura (red, clúster, registro). No gestiona los objetos de Kubernetes, que se aplican con `kubectl`.
- **¿Para qué sirve el estado de Terraform?** Es el registro de lo que Terraform creó. Sin él no puede saber qué borrar en `destroy`.
- **¿Qué es un nodo spot?** Una máquina con descuento que Google puede reclamar. Sirve para ahorrar en un proyecto académico.
- **¿Por qué dos bases separadas?** Cada microservicio es dueño de sus datos. Pedidos guarda una copia histórica del nombre y precio, así que no depende de que Productos esté disponible para consultarse.
- **¿Qué pasa si Productos se cae?** Crear un pedido responde `503`; consultar pedidos existentes sigue funcionando.
- **¿Qué es un rolling update?** Reemplazar los pods de a uno, esperando que el nuevo esté listo, para no interrumpir el servicio.
- **¿Para qué sirven las sondas `live` y `ready`?** `live` indica si el proceso está vivo; `ready`, si puede recibir tráfico (por ejemplo, si la base responde).
- **¿Por qué los Jobs de migración van aparte?** Para que dos réplicas no migren la base a la vez.

## 12. Sección G — Costos y cierre responsable

### Estimación (G1)

Debe existir **antes** del despliegue, en `docs/estimacion-costos.md`, con los precios de la calculadora de Google Cloud para `us-central1`: 2 nodos `e2-small` spot, disco persistente de 10 GB y discos de los nodos, Artifact Registry, 3 balanceadores de carga y la tarifa de administración de GKE.

### Pausar sin destruir

Para dejar de pagar entre sesiones sin perder el clúster ni las imágenes: borrar los balanceadores, apagar las réplicas y llevar los nodos a cero. Queda solo el disco de PostgreSQL y el registro.

In [ ]:
kubectl delete svc frontend productos pedidos -n panaderia-grupo-4

In [ ]:
kubectl scale deployment --all --replicas=0 -n panaderia-grupo-4

In [ ]:
gcloud container clusters resize panaderia-grupo-4 --node-pool panaderia-grupo-4-node-pool --num-nodes 0 --zone us-central1-a --project virtualizacion-proyecto-pr --quiet

Para volver a encender: nodos a 2 y se reaplican los manifiestos (las IP públicas serán nuevas).

In [ ]:
gcloud container clusters resize panaderia-grupo-4 --node-pool panaderia-grupo-4-node-pool --num-nodes 2 --zone us-central1-a --project virtualizacion-proyecto-pr --quiet

In [ ]:
bash deploy/k8s/apply.sh

### Destruir la infraestructura (G2)

⚠️ **Borra el clúster y todo lo que creó Terraform.** Hazlo **después de la defensa**. Antes, se borra el namespace para que no queden balanceadores ni discos creados con `kubectl` fuera del estado de Terraform. **Debe verse:** `Destroy complete! Resources: 6 destroyed.`

In [ ]:
kubectl delete namespace panaderia-grupo-4

In [ ]:
date; terraform -chdir=infra/terraform destroy -auto-approve

### Comprobar que no quedó nada vivo

**Debe verse:** todas las listas vacías, salvo la red `default`.

In [ ]:
gcloud container clusters list --format="value(name)"

In [ ]:
gcloud compute forwarding-rules list --format="value(name)"

In [ ]:
gcloud compute disks list --format="value(name)"

In [ ]:
gcloud artifacts repositories list --format="value(name)"

In [ ]:
gcloud compute networks list --format="value(name)"

### Panel de facturación (G3)

Se captura en la consola web: **Facturación → Informes**, con el rango de fechas del proyecto y agrupado por producto. Debe mostrarse el gasto final. Fecha límite de la evidencia de destrucción: **viernes 13 de noviembre, 23:59**.

## Orden sugerido en una defensa

1. `date; gcloud config list` (siempre primero).
2. `kubectl get all,pvc -n panaderia-grupo-4` (E1).
3. La aplicación en el navegador con `http://IP` (E5).
4. ConfigMap y Secret dentro de un pod (E2, E3).
5. Borrar el pod de PostgreSQL y comprobar que los datos siguen (E4).
6. Escalar a 3 réplicas editando el YAML (F1).
7. Borrar un pod de Productos y ver que se recrea (F2).
8. Rolling update `1.1.0` a `1.1.1` (F3).
9. Logs de un pod (F4).

Practica cada paso hasta poder escribirlo sin mirar este cuaderno.